# Exercise 1 — kelly_fraction and is_stopped_out

Before placing a trade you make two decisions: how much to bet (position sizing) and when to exit if wrong (stop-loss trigger). `kelly_fraction` answers the first question with the Kelly Criterion. `is_stopped_out` answers the second with a single price comparison.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def kelly_fraction(win_rate, avg_win, avg_loss):
    """Kelly Criterion: optimal fraction of capital to risk per trade.

    Formula: f* = win_rate − (1 − win_rate) / (avg_win / avg_loss)

    Returns 0.0 for degenerate inputs (avg_loss ≤ 0, win_rate not in (0,1)).
    Clamp result to [0.0, 1.0].

    Example:
        win_rate=0.6, avg_win=100, avg_loss=100 → 0.6 − 0.4/1 = 0.20
    """
    # TODO: implement
    return 0.0


def is_stopped_out(entry_price, current_price, stop_pct=0.05):
    """True if current_price ≤ entry_price × (1 − stop_pct).

    Returns False for non-positive entry_price.
    """
    # TODO: implement (two lines)
    return False


### Checks

In [ ]:
checks = 0

# 1 — kelly: known example
try:
    k = kelly_fraction(0.6, 100, 100)
    assert abs(k - 0.2) < 1e-9, f"expected 0.2, got {k}"
    checks += 1; print("✅ 1 kelly_fraction(0.6, 100, 100) = 0.20")
except Exception as e:
    print("❌ 1:", e)

# 2 — kelly: different win/loss ratio
try:
    k = kelly_fraction(0.5, 200, 100)   # b=2, f = 0.5 - 0.5/2 = 0.25
    assert abs(k - 0.25) < 1e-9, f"expected 0.25, got {k}"
    checks += 1; print("✅ 2 kelly_fraction(0.5, avg_win=200, avg_loss=100) = 0.25")
except Exception as e:
    print("❌ 2:", e)

# 3 — kelly: edge cases return 0.0
try:
    assert kelly_fraction(0.6, 100, 0)   == 0.0, "avg_loss=0 → 0.0"
    assert kelly_fraction(0.0, 100, 100) == 0.0, "win_rate=0 → 0.0"
    assert kelly_fraction(0.4, 100, 200) == 0.0, "negative kelly → clamped to 0.0"
    checks += 1; print("✅ 3 degenerate inputs return 0.0")
except Exception as e:
    print("❌ 3:", e)

# 4 — is_stopped_out: below stop level → True
try:
    assert is_stopped_out(100.0, 94.0, stop_pct=0.05), "94 ≤ 95 → stopped"
    assert is_stopped_out(100.0, 95.0, stop_pct=0.05), "95 ≤ 95 → stopped (boundary)"
    checks += 1; print("✅ 4 is_stopped_out: price ≤ stop level → True")
except Exception as e:
    print("❌ 4:", e)

# 5 — is_stopped_out: above stop → False; bad entry → False
try:
    assert not is_stopped_out(100.0, 96.0, stop_pct=0.05), "96 > 95 → not stopped"
    assert not is_stopped_out(0.0,   94.0, stop_pct=0.05), "entry=0 → False (guard)"
    assert not is_stopped_out(-1.0,  94.0, stop_pct=0.05), "entry<0 → False"
    checks += 1; print("✅ 5 is_stopped_out: above stop / bad entry → False")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
